In [5]:
!pip -q install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 39.8 MB/s eta 0:00:00


In [6]:
# ============================================================
# EnergyMol-MLIP
# Pilot Molecular Structure Generation
#
# Workflow:
# SMILES
#   ↓
# RDKit parsing
#   ↓
# 3D conformer generation
#   ↓
# MMFF geometry optimization
#   ↓
# XYZ structure
#   ↓
# metadata CSV
#
# All files are saved permanently to Google Drive.
# ============================================================

from pathlib import Path
import csv
import os

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors


# ============================================================
# 1. PROJECT DIRECTORIES
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/MLIP project")

PILOT_DIR = PROJECT_DIR / "pilot_structures"
METADATA_DIR = PROJECT_DIR / "data" / "raw"

# Create directories if they do not already exist
PILOT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("EnergyMol-MLIP | Pilot Structure Generation")
print("=" * 70)

print("\nProject directory:")
print(PROJECT_DIR)

print("\nStructure directory:")
print(PILOT_DIR)

print("\nMetadata directory:")
print(METADATA_DIR)


# ============================================================
# 2. PILOT MOLECULES
# ============================================================

molecules = [
    {
        "name": "benzobithiophene",
        "smiles": "C1(C2=CC3=CC=CC=C3S2)=CC4=CC=CC=C4S1",
        "family": "S/heteroaromatic",
    },
    {
        "name": "BDT",
        "smiles": "C12=CC3=C(C=CS3)C=C1SC=C2",
        "family": "S/heteroaromatic",
    },
    {
        "name": "thiadiazole",
        "smiles": "S1N=NC=C1",
        "family": "S/N heteroaromatic",
    },
]


# ============================================================
# 3. FUNCTION TO GENERATE AND OPTIMIZE 3D STRUCTURE
# ============================================================

def generate_structure(name, smiles, family, output_dir):
    """
    Convert SMILES to a 3D molecular structure,
    optimize using MMFF, and save as XYZ.
    """

    print("\n" + "=" * 70)
    print(f"Molecule: {name}")
    print(f"SMILES:   {smiles}")
    print("=" * 70)

    # --------------------------------------------------------
    # Step 1: Parse SMILES
    # --------------------------------------------------------

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Could not parse SMILES for {name}")

    print("✓ SMILES successfully parsed")

    # --------------------------------------------------------
    # Step 2: Add hydrogens
    # --------------------------------------------------------

    mol = Chem.AddHs(mol)

    total_atoms = mol.GetNumAtoms()
    heavy_atoms = mol.GetNumHeavyAtoms()
    molecular_weight = Descriptors.MolWt(mol)

    print(f"Total atoms:      {total_atoms}")
    print(f"Heavy atoms:      {heavy_atoms}")
    print(f"Molecular weight: {molecular_weight:.4f} g/mol")

    # --------------------------------------------------------
    # Step 3: Generate 3D coordinates
    # --------------------------------------------------------

    params = AllChem.ETKDGv3()
    params.randomSeed = 12345

    embed_status = AllChem.EmbedMolecule(mol, params)

    if embed_status != 0:
        print("⚠ Initial embedding failed.")
        print("Trying random-coordinate embedding...")

        params.useRandomCoords = True

        embed_status = AllChem.EmbedMolecule(mol, params)

    if embed_status != 0:
        raise RuntimeError(
            f"3D coordinate generation failed for {name}"
        )

    print("✓ 3D coordinates generated")

    # --------------------------------------------------------
    # Step 4: MMFF force-field optimization
    # --------------------------------------------------------

    mmff_properties = AllChem.MMFFGetMoleculeProperties(mol)

    if mmff_properties is None:
        print("⚠ MMFF parameters unavailable.")
        print("Trying UFF optimization instead.")

        uff_status = AllChem.UFFOptimizeMolecule(
            mol,
            maxIters=1000
        )

        if uff_status == 0:
            print("✓ UFF geometry optimization converged")
        else:
            print("⚠ UFF optimization did not fully converge")

    else:
        mmff_status = AllChem.MMFFOptimizeMolecule(
            mol,
            mmffVariant="MMFF94",
            maxIters=1000
        )

        if mmff_status == 0:
            print("✓ MMFF geometry optimization converged")
        elif mmff_status == 1:
            print("⚠ MMFF reached maximum iterations")
        else:
            print("⚠ MMFF optimization failed")

    # --------------------------------------------------------
    # Step 5: Save XYZ
    # --------------------------------------------------------

    xyz_path = output_dir / f"{name}.xyz"

    conf = mol.GetConformer()

    with open(xyz_path, "w") as f:

        # Number of atoms
        f.write(f"{total_atoms}\n")

        # Comment line
        f.write(
            f"{name} | family={family} | "
            f"generated_by=RDKit_MMFF\n"
        )

        # Atomic coordinates
        for atom in mol.GetAtoms():

            symbol = atom.GetSymbol()
            position = conf.GetAtomPosition(atom.GetIdx())

            f.write(
                f"{symbol:<3} "
                f"{position.x:16.8f} "
                f"{position.y:16.8f} "
                f"{position.z:16.8f}\n"
            )

    print(f"✓ XYZ saved:")
    print(f"  {xyz_path}")

    # --------------------------------------------------------
    # Step 6: Verify file exists
    # --------------------------------------------------------

    if xyz_path.exists():

        size_kb = xyz_path.stat().st_size / 1024

        print(f"✓ File verified")
        print(f"  Size: {size_kb:.2f} KB")

    else:

        raise FileNotFoundError(
            f"XYZ file was not created: {xyz_path}"
        )

    # --------------------------------------------------------
    # Return metadata
    # --------------------------------------------------------

    return {
        "name": name,
        "smiles": smiles,
        "family": family,
        "total_atoms": total_atoms,
        "heavy_atoms": heavy_atoms,
        "molecular_weight": round(molecular_weight, 4),
        "xyz_file": str(xyz_path),
    }


# ============================================================
# 4. GENERATE ALL PILOT STRUCTURES
# ============================================================

results = []

for molecule in molecules:

    result = generate_structure(
        name=molecule["name"],
        smiles=molecule["smiles"],
        family=molecule["family"],
        output_dir=PILOT_DIR
    )

    results.append(result)


# ============================================================
# 5. SAVE METADATA CSV
# ============================================================

metadata_file = METADATA_DIR / "pilot_molecules.csv"

fieldnames = [
    "name",
    "smiles",
    "family",
    "total_atoms",
    "heavy_atoms",
    "molecular_weight",
    "xyz_file",
]

with open(metadata_file, "w", newline="") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(results)

print("\n" + "=" * 70)
print("PILOT DATASET GENERATION COMPLETE")
print("=" * 70)

print("\nGenerated structures:")

for result in results:
    print(
        f"✓ {result['name']:<20} "
        f"{result['total_atoms']:>3} atoms | "
        f"{result['heavy_atoms']:>2} heavy atoms"
    )

print("\nMetadata saved to:")
print(metadata_file)


# ============================================================
# 6. FINAL DIRECTORY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FILES CURRENTLY IN pilot_structures/")
print("=" * 70)

for file in sorted(PILOT_DIR.iterdir()):

    if file.is_file():

        print(
            f"✓ {file.name:<30} "
            f"{file.stat().st_size / 1024:.2f} KB"
        )

print("\n" + "=" * 70)
print("FILES CURRENTLY IN data/raw/")
print("=" * 70)

for file in sorted(METADATA_DIR.iterdir()):

    if file.is_file():

        print(
            f"✓ {file.name:<30} "
            f"{file.stat().st_size / 1024:.2f} KB"
        )

print("\nAll pilot files are stored permanently in Google Drive.")

EnergyMol-MLIP | Pilot Structure Generation

Project directory:
/content/drive/MyDrive/MLIP project

Structure directory:
/content/drive/MyDrive/MLIP project/pilot_structures

Metadata directory:
/content/drive/MyDrive/MLIP project/data/raw

Molecule: benzobithiophene
SMILES:   C1(C2=CC3=CC=CC=C3S2)=CC4=CC=CC=C4S1
✓ SMILES successfully parsed
Total atoms:      28
Heavy atoms:      18
Molecular weight: 266.3900 g/mol
✓ 3D coordinates generated
✓ MMFF geometry optimization converged
✓ XYZ saved:
  /content/drive/MyDrive/MLIP project/pilot_structures/benzobithiophene.xyz
✓ File verified
  Size: 1.57 KB

Molecule: BDT
SMILES:   C12=CC3=C(C=CS3)C=C1SC=C2
✓ SMILES successfully parsed
Total atoms:      18
Heavy atoms:      12
Molecular weight: 190.2920 g/mol
✓ 3D coordinates generated
✓ MMFF geometry optimization converged
✓ XYZ saved:
  /content/drive/MyDrive/MLIP project/pilot_structures/BDT.xyz
✓ File verified
  Size: 1.02 KB

Molecule: thiadiazole
SMILES:   S1N=NC=C1
✓ SMILES successfully

In [1]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nAbsolute path of pilot_structures:")
print(os.path.abspath("pilot_structures"))

print("\nFiles inside pilot_structures:")
if os.path.exists("pilot_structures"):
    for f in os.listdir("pilot_structures"):
        print(f)
else:
    print("Folder does not exist.")

Current working directory:
/content

Absolute path of pilot_structures:
/content/pilot_structures

Files inside pilot_structures:
Folder does not exist.


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

print(os.listdir('/content/drive/MyDrive'))

['MLIP project', 'IMG-20250305-WA0008.jpg', 'CamScanner 08-03-2025 19.17_1 (1).jpg', 'CamScanner 08-03-2025 19.17_2 (1).jpg', 'Getting started.pdf', 'back page.jpg', '20201219_183133 (1).jpg', '20201219_183133.jpg', '1608387859167 (1).jpg', '1608387703618 (1).jpg', 'computational word form (1).docx', '4ui2t-awf1s (1).pdf', '1608387859167.jpg', '1608387703618.jpg', 'computational word form.docx', '4ui2t-awf1s.pdf', 'sania seminar.pptx', 'sania ismaeel synopsis.docx', 'sania ismaeel.docx', '4TH CHAPTER STRUCTURES.docx', 'sania ismaeel (3) (1).pdf', 'sania ismaeel 2017-ag-566 thesis.docx', '1245 (2).zip', '1245 (1).zip', '1245.zip', 'ANSWERS.docx', 'NEW 1000 MCQs.docx', 'sania ismaeel.pptx', 'semiar computational.pptx', 'ALL MOLECULES', 'periodic_table-color.pdf', 'sania phd seminar.pptx', 'mahira rafiq seminar.pptx', 'arooj seminar.pptx', 'forensic chemistry mass spectrometery.pptx', 'IMG-20220616-WA0009.jpg', 'Resume_21_11_2021_10_38_12_am.pdf', 'IMG-20220624-WA0042.jpg', 'IMG-20220624-

In [4]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/MLIP project")

# Create project folders
folders = [
    "data",
    "data/raw",
    "data/processed",
    "pilot_structures",
    "configs",
    "notebooks",
    "models",
    "results",
    "results/figures",
    "results/metrics",
    "scripts"
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project directory:")
print(PROJECT_DIR)

print("\nProject folders:")
for folder in folders:
    print("✓", PROJECT_DIR / folder)

Project directory:
/content/drive/MyDrive/MLIP project

Project folders:
✓ /content/drive/MyDrive/MLIP project/data
✓ /content/drive/MyDrive/MLIP project/data/raw
✓ /content/drive/MyDrive/MLIP project/data/processed
✓ /content/drive/MyDrive/MLIP project/pilot_structures
✓ /content/drive/MyDrive/MLIP project/configs
✓ /content/drive/MyDrive/MLIP project/notebooks
✓ /content/drive/MyDrive/MLIP project/models
✓ /content/drive/MyDrive/MLIP project/results
✓ /content/drive/MyDrive/MLIP project/results/figures
✓ /content/drive/MyDrive/MLIP project/results/metrics
✓ /content/drive/MyDrive/MLIP project/scripts
